This Notebook is a kaggle tutorial for Japanese kaggle beginners writen in Japanese.

# 5. ????????????????! ?????????????????

???????????????????????????????????????????????????????????????????????????????

????????????????2???????????

- ?????
- ????????????

???????[Grid search](https://qiita.com/yhyhyhjp/items/c81f7cea72a44a7bfd3a), [Bayesian Optimization](https://blog.amedama.jp/entry/2018/08/18/233841), [Hyperopt](https://blog.amedama.jp/entry/hyperopt), [Optuna](https://research.preferred.jp/2018/12/optuna-release/)?????????????????

??[Notebook](https://www.kaggle.com/sishihara/upura-kaggle-tutorial-05-tuning)???????????????????????????????????????????????????????Optuna?????????????????

In [ ]:
# ??????

import numpy as np
import pandas as pd

train = pd.read_csv("../input/titanic/train.csv")
test = pd.read_csv("../input/titanic/test.csv")
gender_submission = pd.read_csv("../input/titanic/gender_submission.csv")

data = pd.concat([train, test], sort=False)

data['Sex'].replace(['male','female'], [0, 1], inplace=True)
data['Embarked'].fillna(('S'), inplace=True)
data['Embarked'] = data['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype(int)
data['Fare'].fillna(np.mean(data['Fare']), inplace=True)
data['Age'].fillna(data['Age'].median(), inplace=True)
data['FamilySize'] = data['Parch'] + data['SibSp'] + 1
data['IsAlone'] = 0
data.loc[data['FamilySize'] == 1, 'IsAlone'] = 1

In [ ]:
data.head()

In [ ]:
delete_columns = ['Name', 'PassengerId', 'Ticket', 'Cabin']
data.drop(delete_columns, axis=1, inplace=True)

train = data[:len(train)]
test = data[len(train):]

y_train = train['Survived']
X_train = train.drop('Survived', axis=1)
X_test = test.drop('Survived', axis=1)

In [ ]:
X_train.head()

??????????????

## LightGBM

X_train?X_train(???)?X_valid(???)???????

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.3, random_state=0, stratify=y_train)

In [ ]:
# ?????????
categorical_features = ['Embarked', 'Pclass', 'Sex']

# ?????

??????LightGBM???????????????????? `objective` ????????????????????????? `default` ?[?](https://lightgbm.readthedocs.io/en/latest/Parameters.html)????????????

In [ ]:
params = {
    'objective': 'binary'
}

????????????????????????????????????

In [ ]:
import lightgbm as lgb


lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)
lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

model = lgb.train(
    params, lgb_train,
    valid_sets=[lgb_train, lgb_eval],
    verbose_eval=10,
    num_boost_round=1000,
    early_stopping_rounds=10
)

y_pred = model.predict(X_test, num_iteration=model.best_iteration)

In [ ]:
y_pred[:10]

??????documentation??[Parameters Tuning](https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html)????????????????????????????????????????????????Tips??????????

??????????????????[For Better Accuracy](https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html#for-better-accuracy)????????

> - Use large max_bin (may be slower)
> - Use small learning_rate with large num_iterations
> - Use large num_leaves (may cause over-fitting)
> - Use bigger training data
> - Try dart

- 1????????`max_bin`???????`default`???255????????300???????
- 2????????`learning_rate`???????`default`???0.1????????0.05???????
- 3????????`num_leaves`???????`default`???31????????40???????

??????????????????????????????????????????????????????????????????????????????????????

???????????????????????[???documentation](https://lightgbm.readthedocs.io/en/latest/Parameters.html)??????????????????????????[???](https://nykergoto.hatenablog.jp/entry/2019/03/29/%E5%8B%BE%E9%85%8D%E3%83%96%E3%83%BC%E3%82%B9%E3%83%86%E3%82%A3%E3%83%B3%E3%82%B0%E3%81%A7%E5%A4%A7%E4%BA%8B%E3%81%AA%E3%83%91%E3%83%A9%E3%83%A1%E3%83%BC%E3%82%BF%E3%81%AE%E6%B0%97%E6%8C%81%E3%81%A1)?LightGBM?????????????????????????????????????

In [ ]:
params = {
    'objective': 'binary',
    'max_bin': 300,
    'learning_rate': 0.05,
    'num_leaves': 40
}

In [ ]:
lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)
lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

model = lgb.train(
    params, lgb_train,
    valid_sets=[lgb_train, lgb_eval],
    verbose_eval=10,
    num_boost_round=1000,
    early_stopping_rounds=10
)

y_pred = model.predict(X_test, num_iteration=model.best_iteration)

In [ ]:
y_pred[:10]

y_pred???????????????????????????????????????????????`valid_1's binary_logloss`? 0.433251 ????????????????????`binary_logloss` ???????????????????

In [ ]:
y_pred = (y_pred > 0.5).astype(int)
y_pred[:10]

In [ ]:
sub = gender_submission

sub['Survived'] = y_pred
sub.to_csv("submission_lightgbm_handtuning.csv", index=False)

sub.head()

LightGBM?????????????????????0.77033?????????????????????????0.75598?????????????????

# Optuna???

????????????????????????????????????????????????????????

- ????????????????????????????????????
- ???????????????????????????????????????????

???????????????????????????????????????????Optuna?????????

Optuna????????????????????`trial.suggest_int()`???????????????? 

?????????`learning_rate`?????????????????????LightGBM?????????`learning_rate`???????????????????????????????????????????????????????????

In [ ]:
# Kaggle?Kernel????
import optuna
from sklearn.metrics import log_loss


def objective(trial):
    params = {
        'objective': 'binary',
        'max_bin': trial.suggest_int('max_bin', 255, 500),
        'learning_rate': 0.05,
        'num_leaves': trial.suggest_int('num_leaves', 32, 128),
    }
    
    lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)
    lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

    model = lgb.train(
        params, lgb_train,
        valid_sets=[lgb_train, lgb_eval],
        verbose_eval=10,
        num_boost_round=1000,
        early_stopping_rounds=10
    )

    y_pred_valid = model.predict(X_valid, num_iteration=model.best_iteration)
    score = log_loss(y_valid, y_pred_valid)
    return score

`n_trials`??????????????????????40???????????????????????

In [ ]:
study = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=0))
study.optimize(objective, n_trials=40)

In [ ]:
study.best_params

??????????????????????????????????????????????????????????????????????

In [ ]:
params = {
    'objective': 'binary',
    'max_bin': study.best_params['max_bin'],
    'learning_rate': 0.05,
    'num_leaves': study.best_params['num_leaves']
}

lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_features)
lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

model = lgb.train(
    params, lgb_train,
    valid_sets=[lgb_train, lgb_eval],
    verbose_eval=10,
    num_boost_round=1000,
    early_stopping_rounds=10
)

y_pred = model.predict(X_test, num_iteration=model.best_iteration)

In [ ]:
y_pred = (y_pred > 0.5).astype(int)

sub['Survived'] = y_pred
sub.to_csv("submission_lightgbm_optuna.csv", index=False)

sub.head()

???????????????????0.77033???????????????????????????????????????????????????????????????????